In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from config import (
    CATEGORY_ROUND0_SUMMARY_PATH,
    CATEGORY_APE_ALL_SUMMARY_PATH,
    BEST_PROMPT_BY_CATEGORY_PATH,
)
from llm_utils import is_valid_prompt_instruction
from prompt_utils import get_category_meta_prompt

Using local model path: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
round0_summary = pd.read_csv(CATEGORY_ROUND0_SUMMARY_PATH)

try:
    ape_summary = pd.read_csv(CATEGORY_APE_ALL_SUMMARY_PATH)
except Exception:
    ape_summary = pd.DataFrame()

print("ROUND0 SHAPE:", round0_summary.shape)
print("ROUND0 COLS:", round0_summary.columns.tolist())

print("APE SHAPE:", ape_summary.shape if not ape_summary.empty else "(0, 0)")
print("APE COLS:", ape_summary.columns.tolist() if not ape_summary.empty else [])

ROUND0 SHAPE: (55, 12)
ROUND0 COLS: ['ape_round', 'parent_prompt_id', 'subcategory', 'prompt_id', 'prompt_family', 'prompt_variant', 'prompt_text', 'mean_proxy_score', 'mean_similarity', 'mean_brevity', 'mean_target_leak', 'n_examples']
APE SHAPE: (93, 11)
APE COLS: ['ape_round', 'parent_prompt_id', 'subcategory', 'prompt_id', 'prompt_family', 'prompt_text', 'mean_proxy_score', 'mean_similarity', 'mean_brevity', 'mean_target_leak', 'selection_status']


In [5]:
required_cols = [
    "subcategory",
    "prompt_text",
    "mean_proxy_score",
    "ape_round",
    "parent_prompt_id",
    "prompt_id",
    "prompt_family",
    "prompt_variant",
]

for col in required_cols:
    if col not in round0_summary.columns:
        round0_summary[col] = pd.NA

if not ape_summary.empty:
    for col in required_cols:
        if col not in ape_summary.columns:
            ape_summary[col] = pd.NA

round0_summary = round0_summary[required_cols].copy()

if ape_summary.empty:
    combined_summary = round0_summary.copy()
else:
    ape_summary = ape_summary[required_cols].copy()
    combined_summary = pd.concat([round0_summary, ape_summary], ignore_index=True)

combined_summary["subcategory"] = combined_summary["subcategory"].fillna("").astype(str).str.strip()
combined_summary["prompt_text"] = combined_summary["prompt_text"].fillna("").astype(str).str.strip()
combined_summary["prompt_variant"] = combined_summary["prompt_variant"].fillna("").astype(str).str.strip()
combined_summary["mean_proxy_score"] = pd.to_numeric(
    combined_summary["mean_proxy_score"], errors="coerce"
).fillna(0.0)

combined_summary = combined_summary[
    (combined_summary["subcategory"] != "") &
    (combined_summary["prompt_text"] != "") &
    (combined_summary["prompt_text"].apply(is_valid_prompt_instruction))
].copy()

print("COMBINED SHAPE AFTER FILTER:", combined_summary.shape)
display(combined_summary.head(10))

COMBINED SHAPE AFTER FILTER: (145, 8)


,subcategory,prompt_text,mean_proxy_score,ape_round,parent_prompt_id,prompt_id,prompt_family,prompt_variant
0,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.613788,0,NaN,交通工具_p01,category_meta,v1_meta_base
1,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.595717,0,NaN,交通工具_p02,category_meta,v2_discriminative
2,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.566154,0,NaN,交通工具_p03,category_meta,v3_life_context
3,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.686101,0,NaN,休閒娛樂_p01,category_meta,v1_meta_base
4,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.667851,0,NaN,休閒娛樂_p03,category_meta,v3_life_context
5,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.600724,0,NaN,休閒娛樂_p02,category_meta,v2_discriminative
6,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.548430,0,NaN,動作_p03,category_meta,v3_life_context
7,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.534476,0,NaN,動作_p01,category_meta,v1_meta_base
8,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.522896,0,NaN,動作_p02,category_meta,v2_discriminative
9,動物,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,0.697142,0,NaN,動物_p02,category_meta,v2_discriminative


In [6]:
# professor-style selection bonus
combined_summary["variant_bonus"] = 0.0

combined_summary.loc[
    combined_summary["prompt_variant"] == "v1_meta_base", "variant_bonus"
] = 0.03

combined_summary.loc[
    combined_summary["prompt_variant"] == "v2_discriminative", "variant_bonus"
] = 0.05

combined_summary.loc[
    combined_summary["prompt_variant"] == "v3_life_context", "variant_bonus"
] = 0.04

combined_summary["selection_score"] = (
    combined_summary["mean_proxy_score"] + combined_summary["variant_bonus"]
)

best_prompt_by_category_df = (
    combined_summary
    .sort_values(["subcategory", "selection_score"], ascending=[True, False])
    .groupby("subcategory", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_prompt_by_category_df)

,subcategory,prompt_text,mean_proxy_score,ape_round,parent_prompt_id,prompt_id,prompt_family,prompt_variant,variant_bonus,selection_score
0,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.595717,0,NaN,交通工具_p02,category_meta,v2_discriminative,0.05,0.645717
1,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.686101,0,NaN,休閒娛樂_p01,category_meta,v1_meta_base,0.03,0.716101
2,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.548430,0,NaN,動作_p03,category_meta,v3_life_context,0.04,0.588430
3,動物,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,0.697142,0,NaN,動物_p02,category_meta,v2_discriminative,0.05,0.747142
4,喝,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.989949,0,NaN,喝_p02,category_meta,v2_discriminative,0.05,1.039949
5,娛樂場所,描述地點的常見景象或功能，避免太抽象的描述。,0.353307,1,娛樂場所_p02,娛樂場所_ape_1_42e5edbc,category_resampled,,0.00,0.353307
6,學校,設計一組針對失語症患者的簡單、具體且容易聯想到答案的提示句列表。,0.635580,3,學校_ape_2_4f193443,學校_ape_3_e7a8d43d,category_resampled,,0.00,0.635580
7,家具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.830629,0,NaN,家具_p02,category_meta,v2_discriminative,0.05,0.880629
8,廚房用品,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.609120,0,NaN,廚房用品_p02,category_meta,v2_discriminative,0.05,0.659120
9,文具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.739974,0,NaN,文具_p02,category_meta,v2_discriminative,0.05,0.789974


In [7]:
# manual override for weak categories
MANUAL_BEST_PROMPT_OVERRIDES = {
    "交通工具": get_category_meta_prompt("交通工具"),
    "交通場所": get_category_meta_prompt("交通場所"),
}

for subcat, prompt_text in MANUAL_BEST_PROMPT_OVERRIDES.items():
    if subcat in best_prompt_by_category_df["subcategory"].values:
        best_prompt_by_category_df.loc[
            best_prompt_by_category_df["subcategory"] == subcat,
            "prompt_text"
        ] = prompt_text
        best_prompt_by_category_df.loc[
            best_prompt_by_category_df["subcategory"] == subcat,
            "prompt_variant"
        ] = "manual_prof_override"

display(best_prompt_by_category_df)

,subcategory,prompt_text,mean_proxy_score,ape_round,parent_prompt_id,prompt_id,prompt_family,prompt_variant,variant_bonus,selection_score
0,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,0.595717,0,NaN,交通工具_p02,category_meta,manual_prof_override,0.05,0.645717
1,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.686101,0,NaN,休閒娛樂_p01,category_meta,v1_meta_base,0.03,0.716101
2,動作,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成...,0.548430,0,NaN,動作_p03,category_meta,v3_life_context,0.04,0.588430
3,動物,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動物詞彙，並生成...,0.697142,0,NaN,動物_p02,category_meta,v2_discriminative,0.05,0.747142
4,喝,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.989949,0,NaN,喝_p02,category_meta,v2_discriminative,0.05,1.039949
5,娛樂場所,描述地點的常見景象或功能，避免太抽象的描述。,0.353307,1,娛樂場所_p02,娛樂場所_ape_1_42e5edbc,category_resampled,,0.00,0.353307
6,學校,設計一組針對失語症患者的簡單、具體且容易聯想到答案的提示句列表。,0.635580,3,學校_ape_2_4f193443,學校_ape_3_e7a8d43d,category_resampled,,0.00,0.635580
7,家具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.830629,0,NaN,家具_p02,category_meta,v2_discriminative,0.05,0.880629
8,廚房用品,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.609120,0,NaN,廚房用品_p02,category_meta,v2_discriminative,0.05,0.659120
9,文具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,0.739974,0,NaN,文具_p02,category_meta,v2_discriminative,0.05,0.789974


In [8]:
#best_prompt_df.to_csv(BEST_PROMPT_PATH, index=False, encoding="utf-8-sig")

#best_prompt_text = best_prompt_df.iloc[0]["prompt_text"]
#with open(BEST_PROMPT_TXT_PATH, "w", encoding="utf-8") as f:
 #   f.write(best_prompt_text)

#print("Saved:", BEST_PROMPT_PATH)
#print("Saved:", BEST_PROMPT_TXT_PATH)

In [9]:
best_prompt_by_category_df.to_csv(
    BEST_PROMPT_BY_CATEGORY_PATH,
    index=False,
    encoding="utf-8-sig"
)
print("Saved:", BEST_PROMPT_BY_CATEGORY_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\best_prompt_by_category.csv


In [10]:
for _, row in best_prompt_by_category_df.iterrows():
    print("=" * 100)
    print("CATEGORY:", row["subcategory"])
    print("PROMPT VARIANT:", row["prompt_variant"])
    print("MEAN PROXY SCORE:", row["mean_proxy_score"])
    print("SELECTION SCORE:", row["selection_score"])
    print("BEST PROMPT:")
    print(row["prompt_text"])
    print()

CATEGORY: 交通工具
PROMPT VARIANT: manual_prof_override
MEAN PROXY SCORE: 0.5957167913516362
SELECTION SCORE: 0.6457167913516363
BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並生成具體、生活化、容易聯想到答案的提示句。

我的最終目標是幫助高齡失語症患者，透過搭乘經驗與用途線索，成功說出目標詞彙。

原則：
1. 優先描述用途、搭乘方式、常見情境。
2. 避免只描述太廣泛的移動概念。
3. 優先使用高齡者熟悉的搭車經驗與日常說法。
4. 避免太書面、太技術化的描述。

提示特徵資料集（交通工具專用）：
●用途：載人、代步、通勤、旅行、上學
●情境：在路上、停靠站牌、上下班、城市裡、出遠門
●辨識特徵：固定路線、很多人一起搭、司機開、有座位、有輪子
●熟悉場景：等車、刷卡、買票、坐著移動

生成限制：
●提示句中不得出現目標詞彙。
●每句長度限於 20 個中文字以內。
●需具體、自然、容易理解。
●盡量指向單一交通工具。

輸出格式：
●每次僅生成一個提示句。
●只輸出提示句本身，不要解釋。

CATEGORY: 休閒娛樂
PROMPT VARIANT: v1_meta_base
MEAN PROXY SCORE: 0.6861010606090227
SELECTION SCORE: 0.7161010606090227
BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一個簡短、具體、生活化且容易聯想到答案的提示句，不可直接說出目標詞。每次只輸出一個提示句，不要解釋。

CATEGORY: 動作
PROMPT VARIANT: v3_life_context
MEAN PROXY SCORE: 0.5484302889549645
SELECTION SCORE: 0.5884302889549645
BEST PROMPT:
你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標動作詞彙，並生成具體、容易理解、方便聯想到答案的提示句。

目標是讓高齡失語症患者透過動作情境、目的或流程聯想

In [11]:
output_txt = PROJECT_ROOT / "data" / "final" / "best_prompts_by_category.txt"

with open(output_txt, "w", encoding="utf-8") as f:
    for _, row in best_prompt_by_category_df.iterrows():
        f.write("=" * 100 + "\n")
        f.write(f"CATEGORY: {row['subcategory']}\n")
        f.write(f"PROMPT VARIANT: {row['prompt_variant']}\n")
        f.write(f"MEAN PROXY SCORE: {row['mean_proxy_score']}\n")
        f.write(f"SELECTION SCORE: {row['selection_score']}\n")
        f.write("BEST PROMPT:\n")
        f.write(str(row["prompt_text"]) + "\n\n")

print("Saved:", output_txt)

Saved: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\best_prompts_by_category.txt
